In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
train_df = pd.read_csv('train.csv', header=0, names=['label', 'title', 'description'])
test_df  = pd.read_csv('test.csv',  header=0, names=['label', 'title', 'description'])

print(f'Train : {train_df.shape[0]} | Test : {test_df.shape[0]}')
train_df.head()

In [ ]:
print(train_df.info())
print(train_df.isnull().sum())

In [ ]:
label_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
train_df['label_name'] = train_df['label'].map(label_names)

sns.countplot(x='label_name', data=train_df, palette='viridis')
plt.title('Distribution des classes')
plt.tight_layout()
plt.show()

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.replace('\\n', ' ')
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\(.*?\)', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

In [ ]:
train_df['text_full'] = train_df['title'] + ' ' + train_df['description']
test_df['text_full']  = test_df['title']  + ' ' + test_df['description']

train_df['text_clean'] = train_df['text_full'].apply(clean_text)
test_df['text_clean']  = test_df['text_full'].apply(clean_text)

train_df[['label', 'text_full', 'text_clean']].head(3)

In [ ]:
train_df = train_df[train_df['text_clean'] != ''].reset_index(drop=True)
test_df  = test_df[test_df['text_clean']  != ''].reset_index(drop=True)

print(f'Train final : {len(train_df)} | Test final : {len(test_df)}')

In [ ]:
all_words = ' '.join(train_df['text_clean']).split()
freq = Counter(all_words).most_common(20)
words, counts = zip(*freq)

plt.figure(figsize=(12, 4))
plt.bar(words, counts, color='steelblue')
plt.title('Top 20 mots (après nettoyage)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
train_df[['label', 'title', 'description', 'text_full', 'text_clean']].to_csv('train_cleaned.csv', index=False)
test_df[['label', 'title', 'description', 'text_full', 'text_clean']].to_csv('test_cleaned.csv', index=False)

print('Sauvegardé : train_cleaned.csv | test_cleaned.csv')